# Full Pinyin syllable objective

Train one objective across five fixed seeds using an existing experiment bundle.


## Configuration

Select input and output paths, device and objective before defining the runtime. Training is disabled by default.


In [ ]:
from pathlib import Path

EXPERIMENT_ID = 'duyu-chinese-lessons-zh-cn-v1'
EXPERIMENT_PATH = Path("/kaggle/input/duyu-chinese-lessons-zh-cn-v1")
OUTPUT_ROOT = Path("/kaggle/working/runs")

EXPECTED_EXPERIMENT_MANIFEST_SHA256 = '719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185'
DEVICE = "cpu"
MODEL_ID = 'syllable'
RUN_REAL_TRAINING = False


### Build provenance

Canonical source hashes and the experiment manifest pin identify this notebook build.


In [ ]:
BUILD_METADATA = {
  "baseline_project_sha": "f7029f129571bc5f2c2546cc86a25a19d3d0aa1f",
  "embedded_source_sha256": "70ec9db354686cdf74195a0c2df47a9a336153540d0d2426b92d0770b26880a6",
  "experiment_id": "duyu-chinese-lessons-zh-cn-v1",
  "experiment_manifest_sha256": "719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185",
  "source_sha256": {
    "__init__.py": "bfa7860bfad7236ce5bbd3aec032161527e7cd8e1b6607c79c15f14853b9b6b2",
    "artifacts.py": "9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d",
    "data.py": "2be23fc5b5a5916c66074dc447143008719ee3c4b1eb2192b73d5f8b30be5989",
    "data_contract.py": "5c526086867ed477286c1f369ca78e1ccd44fa52ed8453846d87a664f2971fa3",
    "experiment.py": "518c606af5b1535d928e3bb3a7425a15db77525ea4bc944f2af74ffa0248ddb0",
    "ids.py": "832b20b5f97f8328150a6efd57847d9cc56378b703117da1bde65759f29c37ab",
    "metrics.py": "10d41722970c5472cfe9c3f09346ae977d95dab520fc8d72a4f26d3e7f935fbc",
    "model.py": "34932db3ea2ac8b2bf3e0d2631e563dcbe22d686c3408a0d113cd499ff3630a1",
    "objectives.py": "2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47",
    "pinyin.py": "6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3",
    "reproducibility.py": "ef59e1eb6453cfb149d11b6a4f2c7227d1aa8a8b7f354fb1a9a9b6e1590296aa",
    "training.py": "e567ed0eef9646479ce3cf50089ea024b1d12946d18e24444cde5dee00e1108d",
    "training_data.py": "6b16cc0d66276f782c9e839ec37fe6dbce703723f079ef706ef1be94a74cb684"
  }
}


## Required packages

Enable installation if the environment lacks these packages. Keep the environment's existing Torch build when available. Pypinyin must match the pinned label contract; restart the kernel after changing packages already imported.


In [ ]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["torch", "numpy", "pypinyin==0.55.0"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("torch", "numpy", "pypinyin"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pypinyin must match the pinned version 0.55.0")


## Dependencies and provenance

Import the required libraries. Source hashes identify the canonical implementation used to generate these cells.


In [ ]:
from __future__ import annotations
import math
import re
from numbers import Real
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
from pypinyin.constants import PINYIN_DICT
from pypinyin.contrib.tone_convert import to_finals, to_initials, to_normal
import hashlib
import json
from pathlib import Path
import copy
import os
import pickle
import random
import shutil
import tempfile
from pathlib import Path, PureWindowsPath
import numpy as np
from functools import lru_cache
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import importlib.metadata
import platform
import statistics
import sys
import uuid
from datetime import datetime, timezone


BASE_MODEL_IDS = ("syllable", "components", "acoustic")


COMBINED_PREFIX = "components-acoustic-lambda-"


def combined_model_id(weight: Real) -> str:
    """Encode a finite coefficient without truncating its float precision."""
    if isinstance(weight, bool) or not isinstance(weight, Real):
        raise ValueError("lambda must be a finite real number")
    try:
        value = float(weight)
    except (OverflowError, ValueError) as exc:
        raise ValueError("lambda must be a finite real number") from exc
    if not math.isfinite(value):
        raise ValueError("lambda must be a finite real number")
    token = str(int(value)) if value.is_integer() and abs(value) < 1e16 else repr(value).replace("e+", "e")
    return COMBINED_PREFIX + token


def parse_model_id(model_id: str) -> tuple[str, float | None]:
    """Return the objective family and the combined acoustic coefficient."""
    if isinstance(model_id, str) and model_id in BASE_MODEL_IDS:
        return model_id, None
    if not isinstance(model_id, str) or not model_id.startswith(COMBINED_PREFIX):
        raise ValueError("unknown objective model_id")
    token = model_id[len(COMBINED_PREFIX):]
    if not re.fullmatch(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?", token):
        raise ValueError("combined model_id must contain a finite numeric lambda")
    weight = float(token)
    if not math.isfinite(weight):
        raise ValueError("lambda must be a finite real number")
    return "components-acoustic", weight


def canonical_model_id(model_id: str) -> str:
    family, weight = parse_model_id(model_id)
    return combined_model_id(weight) if family == "components-acoustic" else family


## Experiment loading

Validate hashes, source provenance, IDS encoding, labels and fixed splits before loading the immutable experiment bundle.


In [ ]:
def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: str | Path, value) -> None:
    Path(path).write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     indent=2) + "\n", encoding="utf-8")


def write_jsonl(path: str | Path, rows) -> None:
    Path(path).write_text("".join(json.dumps(row, ensure_ascii=False, sort_keys=True)
                                  + "\n" for row in rows), encoding="utf-8")


def read_jsonl(path: str | Path) -> list[dict]:
    result = []
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        try:
            value = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"invalid JSONL at line {number}") from exc
        if not isinstance(value, dict):
            raise ValueError(f"JSONL row {number} is not an object")
        result.append(value)
    return result


def safe_artifact(root: str | Path, relative: str) -> Path:
    root = Path(root).resolve()
    rel = Path(relative)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"unsafe artifact path: {relative}")
    path = (root / rel).resolve()
    if path != root and root not in path.parents:
        raise ValueError(f"unsafe artifact path: {relative}")
    cursor = root
    for part in rel.parts:
        cursor = cursor / part
        if cursor.exists() and cursor.is_symlink():
            raise ValueError(f"symlink artifact path: {relative}")
    return path


### Pinyin contract

Canonical syllables and component inventories define all classification targets.


In [ ]:
PINYIN_CONTRACT_VERSION = "pypinyin-tone3-1"


def _require_pypinyin_version():
    installed = version("pypinyin")
    if installed != "0.55.0":
        raise RuntimeError(f"pypinyin 0.55.0 required, found {installed}")
    return installed


_PYPINYIN_VERSION = _require_pypinyin_version()


VALID_SYLLABLES = frozenset(
    to_normal(reading, v_to_u=False).lower()
    for readings in PINYIN_DICT.values()
    for reading in readings.split(",")
)


_INPUT_SYLLABLES = tuple(sorted(base for base in VALID_SYLLABLES if re.fullmatch(r"[a-z]+", base)))


_STRICT_COMPONENTS = tuple(
    (to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False))
    for base in _INPUT_SYLLABLES
)


INITIALS = sorted({initial for initial, final in _STRICT_COMPONENTS if final})


FINALS = sorted({final for _, final in _STRICT_COMPONENTS if final})


TONES = [str(tone) for tone in range(1, 6)]


PARSER_PROVENANCE = {
    "library": "pypinyin", "version": _PYPINYIN_VERSION,
    "api": "pypinyin.contrib.tone_convert", "strict": True,
    "input_format": "lowercase-ascii-tone3", "umlaut_storage": "v",
    "input_alias": "u:", "missing_tone": "5", "empty_final": "reject",
    "inventory_source": "pypinyin.constants.PINYIN_DICT",
    "inventory_sha256": sha256("\n".join(sorted(VALID_SYLLABLES)).encode("utf-8")).hexdigest(),
    "vocabulary_sha256": sha256(repr((INITIALS, FINALS, TONES)).encode("utf-8")).hexdigest(),
}


@dataclass(frozen=True)
class PinyinComponents:
    initial: str
    final: str
    tone: str


def canonical_label(token: str, *, tone_convention="neutral_unmarked") -> str:
    """Accept one lowercase TONE3 syllable, allowing u: and unmarked neutral."""
    if tone_convention not in {"neutral_unmarked", "explicit"}:
        raise ValueError(f"unsupported tone convention: {tone_convention!r}")
    if not isinstance(token, str) or not token:
        raise ValueError(f"Pinyin must be a non-empty string: {token!r}")
    if re.fullmatch(r"(?:[a-z]|u:)+[1-5]?", token) is None:
        raise ValueError(f"invalid Pinyin format (lowercase ASCII TONE3 required): {token!r}")
    canonical = token.replace("u:", "v")
    if canonical[-1] not in TONES:
        if tone_convention == "explicit":
            raise ValueError(f"explicit tone required: {token!r}")
        canonical += "5"
    if re.fullmatch(r"[a-z]+[1-5]", canonical) is None:
        raise ValueError(f"invalid canonical Pinyin format: {token!r}")
    base = canonical[:-1]
    if base not in VALID_SYLLABLES:
        raise ValueError(f"syllable absent from pypinyin dictionary: {token!r}")
    initial = to_initials(base, strict=True)
    final = to_finals(base, strict=True, v_to_u=False)
    if not final or initial not in INITIALS or final not in FINALS:
        raise ValueError(f"unsupported strict Pinyin components (non-empty final required): {token!r}")
    return canonical


def validate_canonical_label(token: str) -> str:
    """Reject noncanonical stored labels instead of silently repairing them."""
    if canonical_label(token, tone_convention="explicit") != token:
        raise ValueError(f"stored Pinyin label is not canonical: {token!r}")
    return token


def parse_components(token: str, *, tone_convention="neutral_unmarked") -> PinyinComponents:
    canonical = canonical_label(token, tone_convention=tone_convention)
    base = canonical[:-1]
    return PinyinComponents(to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False), canonical[-1])


def encode_components(components: PinyinComponents) -> tuple[int, int, int]:
    try:
        return INITIALS.index(components.initial), FINALS.index(components.final), TONES.index(components.tone)
    except ValueError as exc:
        raise ValueError(f"components outside Pinyin vocabulary: {components!r}") from exc


def canonical_labels(tokens, *, tone_convention="neutral_unmarked"):
    return [canonical_label(token, tone_convention=tone_convention) for token in tokens]


def decompose(token: str, *, tone_convention="neutral_unmarked") -> tuple[int, int, int]:
    return encode_components(parse_components(token, tone_convention=tone_convention))


### IDS validation

Reconstruct the pinned character mapping and verify its token vocabulary.


In [ ]:
def sample_id(text, labels):
    return hashlib.sha256(json.dumps([text, labels], ensure_ascii=False,
                                    separators=(",", ":")).encode("utf-8")).hexdigest()

IDS_SCHEMA_VERSION = 1


IDS_RULES_VERSION = "ids-prefix-1"


PAD = "[PAD]"


UNKNOWN = "？"


OPERATORS = {chr(code) for code in range(0x2FF0, 0x2FFC)}


TERNARY_OPERATORS = {"⿲", "⿳"}


UNSUPPORTED_OPERATORS = {chr(code) for code in range(0x2FFC, 0x3000)} | {"㇯"}


EXPECTED_SNAPSHOT = "618dbab8a8ddefb958763c8b4afbaa741a4460de"


EXPECTED_LICENSE = "LGPL-3.0-or-later"


EXPECTED_ATTRIBUTION = "Make Me a Hanzi; dictionary.txt derived from Unihan and CJKlib"


def parse_expression(expression: str) -> tuple[str, ...]:
    if not isinstance(expression, str) or not expression:
        raise ValueError("IDS decomposition must be a non-empty string")
    if any(char in UNSUPPORTED_OPERATORS for char in expression):
        raise ValueError("unsupported IDS operator")
    cursor = 0
    def parse_node() -> None:
        nonlocal cursor
        if cursor >= len(expression):
            raise ValueError("IDS expression has too few operands")
        symbol = expression[cursor]
        cursor += 1
        if symbol in OPERATORS:
            arity = 3 if symbol in TERNARY_OPERATORS else 2
            for _ in range(arity):
                parse_node()
    parse_node()
    if cursor != len(expression):
        raise ValueError("IDS expression has trailing operands")
    return tuple(expression)


def read_dictionary(path: str | Path) -> dict[str, dict]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    mapping: dict[str, dict] = {}
    with path.open("r", encoding="utf-8", newline="") as handle:
        for line_number, line in enumerate(handle, 1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"invalid IDS JSONL at line {line_number}") from exc
            if not isinstance(row, dict):
                raise ValueError(f"IDS line {line_number} must be an object")
            character, decomposition = row.get("character"), row.get("decomposition")
            if not isinstance(character, str) or len(character) != 1:
                raise ValueError(f"IDS line {line_number} character must be one code point")
            if character in mapping:
                raise ValueError(f"duplicate IDS character: {character}")
            if not isinstance(decomposition, str) or not decomposition:
                raise ValueError(f"empty or invalid IDS decomposition for {character}")
            parsed = parse_expression(decomposition)
            if UNKNOWN in decomposition:
                tokens, mode = (character,), "character_fallback"
            else:
                tokens, mode = parsed, "decomposition"
            mapping[character] = {"decomposition": decomposition, "tokens": list(tokens), "mode": mode}
    if not mapping:
        raise ValueError("IDS dictionary is empty")
    return dict(sorted(mapping.items(), key=lambda item: tuple(map(ord, item[0]))))


def validate_pin(spec: dict) -> tuple[Path, str]:
    expected = {"path", "sha256", "snapshot_commit", "license", "attribution"}
    if not isinstance(spec, dict) or set(spec) != expected:
        raise ValueError("IDS pin must contain path, sha256, snapshot_commit, license, attribution")
    path = Path(spec["path"]).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = sha256_file(path)
    if digest != spec["sha256"]:
        raise ValueError("IDS dictionary checksum mismatch")
    if spec["snapshot_commit"] != EXPECTED_SNAPSHOT:
        raise ValueError("IDS snapshot commit mismatch")
    if spec["license"] != EXPECTED_LICENSE or not isinstance(spec["attribution"], str) or not spec["attribution"].strip():
        raise ValueError("IDS license or attribution mismatch")
    return path, digest


def build_mapping(spec: dict, *, dictionary_path: str | Path | None = None) -> tuple[dict, dict, dict]:
    if dictionary_path is None:
        path, digest = validate_pin(spec)
        records_path = path
    else:
        expected = {"path", "sha256", "snapshot_commit", "license", "attribution"}
        if not isinstance(spec, dict) or set(spec) != expected:
            raise ValueError("IDS pin must contain path, sha256, snapshot_commit, license, attribution")
        records_path = Path(dictionary_path).resolve()
        if not records_path.is_file() or sha256_file(records_path) != spec["sha256"]:
            raise ValueError("bundled IDS dictionary checksum mismatch")
        if spec["snapshot_commit"] != EXPECTED_SNAPSHOT or spec["license"] != EXPECTED_LICENSE or not str(spec["attribution"]).strip():
            raise ValueError("IDS source provenance mismatch")
        # The source path is historical provenance, not a location on this host.
        path, digest = spec["path"], spec["sha256"]
    records = read_dictionary(records_path)
    tokens = sorted({token for record in records.values() for token in record["tokens"]}, key=ord)
    vocabulary = [PAD, *tokens]
    stoi = {token: index for index, token in enumerate(vocabulary)}
    encoded = {character: {**record, "ids": [stoi[token] for token in record["tokens"]]}
               for character, record in records.items()}
    provenance = {"path": str(path), "sha256": digest, "snapshot_commit": spec["snapshot_commit"],
                  "license": spec["license"], "attribution": spec["attribution"]}
    mapping_artifact = {"schema_version": IDS_SCHEMA_VERSION, "rules_version": IDS_RULES_VERSION,
                        "provenance": provenance, "characters": encoded}
    vocabulary_artifact = {"schema_version": IDS_SCHEMA_VERSION, "rules_version": IDS_RULES_VERSION,
                          "tokens": vocabulary, "stoi": stoi}
    return mapping_artifact, vocabulary_artifact, provenance


def encode_text(text: str, mapping: dict, vocabulary: dict) -> tuple[list[list[int]], list[int], str | None]:
    sequences, lengths = [], []
    for character in text:
        if character not in mapping["characters"]:
            raise ValueError(f"character is absent from IDS mapping: {character}")
        record = mapping["characters"][character]
        if len(record["ids"])!=len(record["tokens"]) or any(type(i) is not int or i<=0 or i>=len(vocabulary["tokens"]) or vocabulary["tokens"][i]!=token for token,i in zip(record["tokens"],record["ids"])):
            raise ValueError(f"invalid IDS token ID in mapping: {character}")
        sequences.append(list(record["ids"]))
        lengths.append(len(record["ids"]))
    return sequences, lengths, None


def content_hash(value: dict) -> str:
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


### Bundle schema and loader

Check the stored training protocol and recover samples with aligned component, syllable and acoustic targets.


In [ ]:
PREPARED_SHA256 = "c562722d2a866f4f1904c746c86c5863a310a66ee3875d86fc36f254fd2d7c10"


SCHEMA_VERSION = 3


DEFAULT_MODEL_IDS = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]


REQUIRED_FILES = {
    "samples.jsonl", "ids/mapping.json", "ids/vocabulary.json", "ids/dictionary.jsonl",
    "ids/COPYING", "ids/LGPL", "acoustic_targets.npy", "vocab.json", "splits.json", "config.json",
    "provenance/prepared_manifest.json", "provenance/processed_manifest.json",
    "provenance/processed_config.json", "provenance/acoustic_manifest.json",
    "provenance/sound_inventory.jsonl", "provenance/prepared_corpus.jsonl",
    "provenance/prepared_acoustic_targets.pkl",
}


_ARCH = {
    "embedding_dim": 64, "ids_bigru_hidden_per_direction": 32, "ids_bigru_layers": 1,
    "sentence_bilstm_hidden_per_direction": 64, "sentence_bilstm_layers": 1, "context_dim": 128,
    "classification_hidden_dim": 64, "classification_dropout": 0.5,
}


_TRAIN = {"optimizer": "Adam", "learning_rate": 0.001, "weight_decay": 0.0, "batch_size": 32,
          "epochs": 20, "scheduler": None, "early_stopping": False, "gradient_clipping": None,
          "cosine_epsilon": 1e-8}


def default_experiment_config(acoustic_dim=1024):
    """Return a fresh config with the agreed protocol and current input pin."""
    return {"schema_version": SCHEMA_VERSION, "prepared_manifest_sha256": PREPARED_SHA256,
            "split_ratios": [0.8, 0.1, 0.1], "split_seed": 0, "training_seeds": [0, 1, 2, 3, 4],
            "architecture": {**_ARCH, "acoustic_dim": acoustic_dim},
            "training": copy.deepcopy(_TRAIN), "model_ids": list(DEFAULT_MODEL_IDS)}


def _validate_config(value, acoustic_dim):
    expected = default_experiment_config(acoustic_dim)
    if not isinstance(value, dict) or set(value) != set(expected):
        raise ValueError("invalid experiment config keys")
    cfg = copy.deepcopy(value)
    def exact_type_value(actual, wanted):
        if type(actual) is not type(wanted): return False
        if isinstance(wanted, dict): return set(actual) == set(wanted) and all(exact_type_value(actual[k], wanted[k]) for k in wanted)
        if isinstance(wanted, list): return len(actual) == len(wanted) and all(exact_type_value(a, b) for a, b in zip(actual, wanted))
        return actual == wanted
    if (type(cfg["schema_version"]) is not int or cfg["schema_version"] != SCHEMA_VERSION or
        not exact_type_value(cfg["architecture"], expected["architecture"]) or
        not exact_type_value(cfg["training"], _TRAIN)):
        raise ValueError("experiment config conflicts with the training protocol")
    model_ids = cfg["model_ids"]
    if not isinstance(model_ids, list) or not model_ids:
        raise ValueError("model_ids must be a nonempty list")
    canonical_ids = [canonical_model_id(model_id) for model_id in model_ids]
    if len(set(canonical_ids)) != len(canonical_ids):
        raise ValueError("model_ids must identify distinct objectives")
    pin = cfg["prepared_manifest_sha256"]
    if not isinstance(pin, str) or len(pin) != 64 or any(c not in "0123456789abcdef" for c in pin):
        raise ValueError("prepared_manifest_sha256 must be a lowercase SHA-256")
    ratios = cfg["split_ratios"]
    if (not isinstance(ratios, list) or len(ratios) != 3 or
        any(type(x) not in (int, float) or not math.isfinite(x) or x <= 0 for x in ratios) or
        not math.isclose(sum(ratios), 1.0, rel_tol=0.0, abs_tol=1e-9)):
        raise ValueError("split_ratios must contain three finite positive values summing to one")
    if type(cfg["split_seed"]) is not int or cfg["split_seed"] != 0:
        raise ValueError("split_seed must be the fixed protocol value 0")
    seeds = cfg["training_seeds"]
    if (not isinstance(seeds, list) or len(seeds) != 5 or any(type(s) is not int for s in seeds)
        or seeds != [0, 1, 2, 3, 4]):
        raise ValueError("training_seeds must be the fixed protocol values [0, 1, 2, 3, 4]")
    return cfg


def _split_ids(ids, seed, ratios):
    values = sorted(ids)
    random.Random(seed).shuffle(values)
    n = len(values)
    train_end = int(n * ratios[0])
    validation_end = train_end + int(n * ratios[1])
    result = {"train": values[:train_end], "validation": values[train_end:validation_end], "test": values[validation_end:]}
    if any(not group for group in result.values()):
        raise ValueError("train, validation, and test splits must all be nonempty")
    return result


def _safe_id(value):
    return (isinstance(value, str) and bool(value) and value not in {".", ".."}
            and "/" not in value and "\\" not in value and Path(value).name == value
            and PureWindowsPath(value).name == value)


def _validate_semantics(root, manifest, config, rows, splits, vocab, matrix):
    if manifest.get("pinyin_contract") != PINYIN_CONTRACT_VERSION or manifest.get("parser_provenance") != PARSER_PROVENANCE or manifest.get("ids_rules") != IDS_RULES_VERSION:
        raise ValueError("unsupported experiment provenance contract")
    pin = manifest.get("prepared_manifest_sha256")
    if sha256_file(root / "provenance/prepared_manifest.json") != pin or pin != config["prepared_manifest_sha256"]:
        raise ValueError("prepared provenance pin mismatch")
    prepared = json.loads((root / "provenance/prepared_manifest.json").read_text(encoding="utf-8"))
    if prepared.get("dataset_id") != manifest.get("prepared_dataset_id"):
        raise ValueError("prepared dataset identity mismatch")
    processed_path = root / "provenance/processed_manifest.json"
    processed = json.loads(processed_path.read_text(encoding="utf-8"))
    acoustic_path = root / "provenance/acoustic_manifest.json"
    acoustic_manifest = json.loads(acoustic_path.read_text(encoding="utf-8"))
    inventory_path = root / "provenance/sound_inventory.jsonl"
    inventory = read_jsonl(inventory_path)
    inventory_hash = sha256_file(inventory_path)
    corpus_path = root / "provenance/prepared_corpus.jsonl"
    acoustic_targets_path = root / "provenance/prepared_acoustic_targets.pkl"
    if (prepared.get("artifacts", {}).get("corpus.jsonl") != sha256_file(corpus_path) or
        prepared.get("artifacts", {}).get("acoustic/targets.pkl") != sha256_file(acoustic_targets_path) or
        prepared.get("processed_corpus_manifest_sha256") != sha256_file(processed_path) or
        prepared.get("acoustic_bundle_manifest_sha256") != sha256_file(acoustic_path) or
        processed.get("sources", {}).get("sound_inventory_sha256") != inventory_hash or
        acoustic_manifest.get("inventory_sha256") != inventory_hash or
        processed.get("artifacts", {}).get("config.json") != sha256_file(root / "provenance/processed_config.json")):
        raise ValueError("copied upstream provenance does not agree")
    if prepared.get("artifacts", {}).get("ids/mapping.json") != sha256_file(root / "ids/mapping.json"):
        raise ValueError("prepared IDS mapping provenance mismatch")
    if prepared.get("artifacts", {}).get("ids/vocabulary.json") != sha256_file(root / "ids/vocabulary.json"):
        raise ValueError("prepared IDS vocabulary provenance mismatch")
    if prepared.get("artifacts", {}).get("ids/dictionary.jsonl") != sha256_file(root / "ids/dictionary.jsonl"):
        raise ValueError("prepared IDS dictionary provenance mismatch")
    for name, digest in manifest["provenance"].items():
        if sha256_file(safe_artifact(root, name)) != digest:
            raise ValueError(f"provenance checksum mismatch: {name}")
    if set(manifest["provenance"]) != {n for n in manifest["artifacts"] if n.startswith("provenance/")}:
        raise ValueError("provenance file set mismatch")

    original_vocab = json.loads((root / "ids/vocabulary.json").read_text(encoding="utf-8"))
    original_mapping = json.loads((root / "ids/mapping.json").read_text(encoding="utf-8"))
    bundled_vocab = vocab["ids"]
    if bundled_vocab != original_vocab or vocab["ids_mapping"] != original_mapping:
        raise ValueError("duplicated IDS mapping or vocabulary mismatch")
    if original_vocab.get("tokens", [None])[0] != "[PAD]" or original_vocab.get("stoi", {}).get("[PAD]") != 0:
        raise ValueError("IDS PAD token must have ID zero")
    ids_cfg = prepared.get("ids_provenance")
    rebuilt_mapping, rebuilt_vocab, _ = build_mapping(ids_cfg, dictionary_path=root / "ids/dictionary.jsonl")
    if rebuilt_mapping != original_mapping or rebuilt_vocab != original_vocab:
        raise ValueError("bundled IDS mapping or vocabulary fails reconstruction")

    tokens = vocab.get("acoustic_tokens")
    if (not isinstance(tokens, list) or not tokens or tokens != sorted(tokens) or len(tokens) != len(set(tokens))
        or vocab.get("syllables") != tokens or vocab.get("labels") != {"initials": INITIALS, "finals": FINALS, "tones": TONES}):
        raise ValueError("invalid acoustic or syllable vocabulary")
    inventory_labels = [row.get("label") for row in inventory]
    if sorted(inventory_labels) != tokens or len(inventory_labels) != len(set(inventory_labels)):
        raise ValueError("acoustic inventory does not match full syllable vocabulary")
    key_ids = {label: i for i, label in enumerate(tokens)}
    for label in tokens:
        validate_canonical_label(label)
    if matrix.shape != (len(tokens), config["architecture"]["acoustic_dim"]) or not np.isfinite(matrix).all() or np.any(np.linalg.norm(matrix.astype(np.float64), axis=1) <= 1e-12):
        raise ValueError("invalid acoustic target matrix")
    with acoustic_targets_path.open("rb") as handle:
        pinned_acoustic = pickle.load(handle)
    pinned_keys = sorted(pinned_acoustic)
    pinned_matrix = np.stack([np.asarray(pinned_acoustic[k], dtype=np.float32) for k in pinned_keys])
    if pinned_keys != tokens or pinned_matrix.shape != matrix.shape or not np.array_equal(pinned_matrix, matrix):
        raise ValueError("acoustic matrix does not reproduce pinned prepared targets")
    mapping_chars = original_mapping.get("characters", {})
    id_count = len(original_vocab["tokens"])
    seen_ids = []
    seen_sources = set()
    for row in rows:
        required = {"component_lengths", "ids_sequences", "labels", "positions", "sample_id", "sentence_length", "source_ids", "text", "component_labels", "syllable_ids", "acoustic_ids"}
        if set(row) != required:
            raise ValueError("sample keys do not match experiment schema")
        text, labels = row["text"], row["labels"]
        n = len(text) if isinstance(text, str) else -1
        if n <= 0 or type(row["sentence_length"]) is not int or row["sentence_length"] != n:
            raise ValueError("invalid sample text or sentence length")
        if row["sample_id"] != sample_id(text, labels):
            raise ValueError("sample ID does not match text and labels")
        if (not isinstance(row["positions"], list) or any(type(pos) is not int for pos in row["positions"])
            or row["positions"] != list(range(n)) or not isinstance(labels, list) or len(labels) != n
            or not isinstance(row["ids_sequences"], list) or len(row["ids_sequences"]) != n
            or not isinstance(row["component_lengths"], list) or len(row["component_lengths"]) != n):
            raise ValueError("sample alignment mismatch")
        if (not isinstance(row["source_ids"], list) or not row["source_ids"] or
            any(not isinstance(s, str) or not s for s in row["source_ids"]) or len(set(row["source_ids"])) != len(row["source_ids"])):
            raise ValueError("invalid sample source IDs")
        if seen_sources.intersection(row["source_ids"]):
            raise ValueError("source IDs occur in more than one sample")
        seen_sources.update(row["source_ids"])
        expected_components, expected_syllables, expected_sequences = [], [], []
        for char, label, sequence, length in zip(text, labels, row["ids_sequences"], row["component_lengths"]):
            canonical = validate_canonical_label(label)
            expected_components.append(list(decompose(canonical)))
            if canonical not in key_ids:
                raise ValueError(f"missing acoustic target for label: {canonical}")
            expected_syllables.append(key_ids[canonical])
            if not isinstance(sequence, list) or not sequence or type(length) is not int or length != len(sequence):
                raise ValueError("invalid IDS sequence or component length")
            if any(type(token) is not int or token <= 0 or token >= id_count for token in sequence):
                raise ValueError("IDS token ID outside vocabulary or padding")
            char_entry = mapping_chars.get(char)
            if char_entry is None or sequence != char_entry.get("ids"):
                raise ValueError("IDS sequence does not match character mapping")
            expected_sequences.append(sequence)
        if (not isinstance(row["component_labels"], list) or
            any(not isinstance(x, list) or any(type(i) is not int for i in x) for x in row["component_labels"]) or
            row["component_labels"] != expected_components):
            raise ValueError("component targets mismatch canonical Pinyin")
        if any(type(i) is not int for i in row["syllable_ids"] + row["acoustic_ids"]):
            raise ValueError("syllable/acoustic IDs must be integers")
        if row["syllable_ids"] != expected_syllables or row["acoustic_ids"] != expected_syllables:
            raise ValueError("syllable/acoustic targets mismatch canonical Pinyin")
        seen_ids.append(row["sample_id"])
    if seen_ids != sorted(seen_ids) or len(seen_ids) != len(set(seen_ids)):
        raise ValueError("sample ID order or uniqueness invalid")
    source_rows = read_jsonl(corpus_path)
    base_fields = {"component_lengths", "ids_sequences", "labels", "positions", "sample_id", "sentence_length", "source_ids", "text"}
    projected_rows = [{key: row[key] for key in base_fields} for row in rows]
    if projected_rows != source_rows:
        raise ValueError("experiment samples do not match pinned prepared corpus")

    if set(splits) != {"train", "validation", "test"}:
        raise ValueError("split names are invalid")
    split_ids = [sid for name in ("train", "validation", "test") for sid in splits[name]]
    if any(not splits[k] for k in splits) or len(split_ids) != len(set(split_ids)) or set(split_ids) != set(seen_ids):
        raise ValueError("split membership is incomplete or duplicated")
    if splits != _split_ids(seen_ids, config["split_seed"], config["split_ratios"]):
        raise ValueError("splits do not match deterministic config")
    counts = manifest.get("counts")
    expected_counts = {"samples": len(rows), "positions": sum(row["sentence_length"] for row in rows),
                       "acoustic_targets": len(tokens), "acoustic_dim": matrix.shape[1]}
    if counts != expected_counts:
        raise ValueError("experiment counts mismatch")


def validate_experiment(path):
    root = Path(path).resolve()
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    if set(manifest) != {"bundle_type", "schema_version", "experiment_id", "pinyin_contract", "parser_provenance", "ids_rules", "prepared_manifest_sha256", "prepared_dataset_id", "provenance", "counts", "artifacts"}:
        raise ValueError("invalid experiment manifest keys")
    if manifest.get("bundle_type") != "experiment_dataset" or manifest.get("schema_version") != SCHEMA_VERSION or not _safe_id(manifest.get("experiment_id")):
        raise ValueError("unsupported experiment bundle")
    artifacts = manifest["artifacts"]
    if set(artifacts) != REQUIRED_FILES:
        raise ValueError("experiment required artifact set mismatch")
    actual = {str(p.relative_to(root)).replace("\\", "/") for p in root.rglob("*") if p.is_file() and p.name != "manifest.json"}
    if actual != set(artifacts):
        raise ValueError("experiment contains untracked or missing artifacts")
    for relative, digest in artifacts.items():
        p = safe_artifact(root, relative)
        if not p.is_file() or not isinstance(digest, str) or sha256_file(p) != digest:
            raise ValueError(f"artifact checksum mismatch: {relative}")
    config = json.loads((root / "config.json").read_text(encoding="utf-8"))
    matrix = np.load(root / "acoustic_targets.npy", allow_pickle=False)
    config = _validate_config(config, matrix.shape[1] if matrix.ndim == 2 else -1)
    if manifest.get("prepared_manifest_sha256") != config["prepared_manifest_sha256"]:
        raise ValueError("manifest and config input pins disagree")
    vocabulary = json.loads((root / "vocab.json").read_text(encoding="utf-8"))
    splits = json.loads((root / "splits.json").read_text(encoding="utf-8"))
    rows = read_jsonl(root / "samples.jsonl")
    _validate_semantics(root, manifest, config, rows, splits, vocabulary, matrix)
    return manifest


def load_experiment(path):
    """Load a portable bundle; samples are keyed by sample_id."""
    manifest = validate_experiment(path)
    root = Path(path).resolve()
    return {"root": root, "manifest": manifest, "config": json.loads((root / "config.json").read_text(encoding="utf-8")),
            "vocab": json.loads((root / "vocab.json").read_text(encoding="utf-8")),
            "splits": json.loads((root / "splits.json").read_text(encoding="utf-8")),
            "samples": {row["sample_id"]: row for row in read_jsonl(root / "samples.jsonl")},
            "acoustic_targets": np.load(root / "acoustic_targets.npy", allow_pickle=False)}


## Batch construction

Pad IDS tokens and sentence positions independently. Real lengths and the character mask exclude padding from recurrent computation and losses.


In [ ]:
class ExperimentDataset(Dataset):
    def __init__(self, sample_ids, samples):
        self.sample_ids = list(sample_ids)
        self.samples = samples

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        return self.samples[self.sample_ids[index]]


def collate_experiment(samples):
    if not samples:
        raise ValueError("cannot collate an empty batch")
    batch_size = len(samples)
    max_sentence = max(sample["sentence_length"] for sample in samples)
    max_component = max(max(sample["component_lengths"]) for sample in samples)
    ids_sequences = torch.zeros((batch_size, max_sentence, max_component), dtype=torch.long)
    component_lengths = torch.zeros((batch_size, max_sentence), dtype=torch.long)
    component_labels = torch.full((batch_size, max_sentence, 3), -100, dtype=torch.long)
    syllable_ids = torch.full((batch_size, max_sentence), -100, dtype=torch.long)
    acoustic_ids = torch.full((batch_size, max_sentence), -1, dtype=torch.long)
    sentence_lengths = torch.empty(batch_size, dtype=torch.long)
    for row_index, sample in enumerate(samples):
        n = sample["sentence_length"]
        sentence_lengths[row_index] = n
        for position, (sequence, length) in enumerate(zip(sample["ids_sequences"], sample["component_lengths"])):
            if length != len(sequence) or length <= 0:
                raise ValueError("sample contains invalid IDS component length")
            ids_sequences[row_index, position, :length] = torch.as_tensor(sequence, dtype=torch.long)
            component_lengths[row_index, position] = length
        component_labels[row_index, :n] = torch.as_tensor(sample["component_labels"], dtype=torch.long)
        syllable_ids[row_index, :n] = torch.as_tensor(sample["syllable_ids"], dtype=torch.long)
        acoustic_ids[row_index, :n] = torch.as_tensor(sample["acoustic_ids"], dtype=torch.long)
    char_mask = component_lengths > 0
    return {
        "ids_sequences": ids_sequences, "component_lengths": component_lengths,
        "sentence_lengths": sentence_lengths, "char_mask": char_mask,
        "component_labels": component_labels, "syllable_ids": syllable_ids,
        "acoustic_ids": acoustic_ids,
        "sample_ids": [sample["sample_id"] for sample in samples],
        "texts": [sample["text"] for sample in samples],
        "positions": [sample["positions"] for sample in samples],
    }


def create_loaders(experiment: dict, seed: int, batch_size: int | None = None, num_workers: int = 0):
    manifest = experiment.get("manifest", {})
    if manifest.get("bundle_type") != "experiment_dataset" or manifest.get("schema_version") != 3:
        raise ValueError("training loaders require a validated schema-3 experiment dataset")
    if type(seed) is not int or seed < 0:
        raise ValueError("loader seed must be a nonnegative integer")
    batch = batch_size or experiment["config"].get("training", {}).get("batch_size", 32)
    if type(batch) is not int or batch <= 0 or type(num_workers) is not int or num_workers < 0:
        raise ValueError("batch_size must be positive and num_workers nonnegative")
    result = {}
    for stream, split in enumerate(("train", "validation", "test"), start=1):
        dataset = ExperimentDataset(experiment["splits"][split], experiment["samples"])
        result[split] = DataLoader(dataset, batch_size=batch, shuffle=split == "train",
                                   generator=make_generator(seed, stream), num_workers=num_workers,
                                   collate_fn=collate_experiment)
    return result


## IDS character encoder

Embed IDS tokens in 64 dimensions. Concatenate the terminal states of a bidirectional GRU to produce one 64-dimensional vector per character.


In [ ]:
class IDSCharacterEncoder(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.bigru = nn.GRU(64, 32, num_layers=1, bidirectional=True, batch_first=True)

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        valid = lengths.reshape(-1) > 0
        if not bool(valid.any()):
            return self.embedding.weight.new_zeros((lengths.numel(), 64))
        selected_ids = token_ids.reshape(-1, token_ids.shape[-1])[valid]
        selected_lengths = lengths.reshape(-1)[valid].to(device="cpu", dtype=torch.long)
        embedded = self.embedding(selected_ids)
        packed = pack_padded_sequence(embedded, selected_lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.bigru(packed)
        character_vectors = torch.cat((hidden[-2], hidden[-1]), dim=-1)
        result = self.embedding.weight.new_zeros((lengths.numel(), 64))
        return result.index_copy(0, valid.nonzero(as_tuple=False).flatten(), character_vectors).reshape(*lengths.shape, 64)


## Context encoder and readouts

A bidirectional LSTM produces 128-dimensional contextual states. Classification heads use dropout; the affine acoustic head and shared encoder do not.


In [ ]:
class SharedEncoder(nn.Module):
    """Encode variable-length IDS for each character, then full sentence context."""
    context_dim = 128

    def __init__(self, ids_vocab_size: int):
        super().__init__()
        self.ids = IDSCharacterEncoder(ids_vocab_size)
        self.sentence = nn.LSTM(64, 64, num_layers=1, bidirectional=True, batch_first=True)

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        token_ids = batch["ids_sequences"]
        component_lengths = batch["component_lengths"]
        sentence_lengths = batch["sentence_lengths"].to(dtype=torch.long)
        mask = batch["char_mask"].to(dtype=torch.bool)
        if token_ids.ndim != 3 or component_lengths.shape != token_ids.shape[:2] or mask.shape != token_ids.shape[:2]:
            raise ValueError("invalid padded IDS batch shapes")
        if not torch.equal(mask, component_lengths > 0):
            raise ValueError("character mask and IDS lengths disagree")
        if not bool((sentence_lengths > 0).all()) or not torch.equal(sentence_lengths.cpu(), mask.sum(1).cpu()):
            raise ValueError("sentence lengths disagree with character mask")
        char_vectors = self.ids(token_ids, component_lengths)
        packed = pack_padded_sequence(char_vectors, sentence_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.sentence(packed)
        contextual, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=token_ids.shape[1])
        return contextual, mask

    def forward(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encode(batch)


class ClassifierHead(nn.Module):
    def __init__(self, output_dim: int):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, output_dim))

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        return self.layers(hidden)


class TrainingModel(nn.Module):
    def __init__(self, encoder: SharedEncoder, model_id: str, vocab: dict, acoustic_dim: int):
        super().__init__()
        model_id = canonical_model_id(model_id)
        family, weight = parse_model_id(model_id)
        self.encoder = encoder
        self.model_id = model_id
        self.lambda_cosine = weight
        self.context_dim = 128
        labels = vocab["labels"]
        self.component_heads = nn.ModuleDict()
        if family in {"components", "components-acoustic"}:
            for name in ("initials", "finals", "tones"):
                self.component_heads[name] = ClassifierHead(len(labels[name]))
        self.syllable_head = ClassifierHead(len(vocab["syllables"])) if family == "syllable" else None
        self.acoustic_head = nn.Linear(128, int(acoustic_dim)) if family in {"acoustic", "components-acoustic"} else None

    def encode(self, batch: dict) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encoder.encode(batch)

    def forward(self, batch: dict) -> dict:
        hidden, mask = self.encoder.encode(batch)
        result = {"hidden": hidden, "mask": mask}
        if self.component_heads:
            result["component_logits"] = {name: head(hidden) for name, head in self.component_heads.items()}
        if self.syllable_head is not None:
            result["syllable_logits"] = self.syllable_head(hidden)
        if self.acoustic_head is not None:
            result["acoustic_prediction"] = self.acoustic_head(hidden)
        return result


## Objectives and metrics

Average losses over real character positions: syllable CE, mean component CE, acoustic cosine loss, or their weighted combination. Invalid initial-final predictions are diagnostic only.


In [ ]:
@lru_cache(maxsize=1)
def valid_initial_final_pairs() -> frozenset[tuple[int, int]]:
    """Return pairs represented by labels accepted by the strict Pinyin API."""
    pairs = set()
    for base in sorted(VALID_SYLLABLES):
        try:
            canonical = canonical_label(base + "1", tone_convention="explicit")
            initial_id, final_id, _ = decompose(canonical, tone_convention="explicit")
        except (ValueError, TypeError, IndexError):
            continue
        pairs.add((initial_id, final_id))
    if not pairs:
        raise RuntimeError("strict Pinyin inventory contains no valid initial/final pairs")
    return frozenset(pairs)

COSINE_EPSILON = 1e-8


def batch_objective(outputs: dict, batch: dict, acoustic_targets: torch.Tensor, model_id: str) -> dict:
    family, weight = parse_model_id(model_id)
    mask = outputs["mask"]
    count = int(mask.sum().item())
    if count <= 0 or not torch.equal(mask, batch["acoustic_ids"] >= 0):
        raise ValueError("batch target masks are empty or misaligned")
    values = {"characters": count, "objective_sum": outputs["hidden"].new_zeros(())}
    if family == "syllable":
        target = batch["syllable_ids"][mask]
        loss = F.cross_entropy(outputs["syllable_logits"][mask], target, reduction="sum")
        values["syllable_ce_sum"] = loss
        values["objective_sum"] = values["objective_sum"] + loss
        values["syllable_correct"] = int((outputs["syllable_logits"][mask].argmax(-1) == target).sum().item())
    if family in {"components", "components-acoustic"}:
        labels = batch["component_labels"][mask]
        ce_sums = []
        for index, name in enumerate(("initials", "finals", "tones")):
            logits = outputs["component_logits"][name][mask]
            target = labels[:, index]
            loss = F.cross_entropy(logits, target, reduction="sum")
            values[f"{name}_ce_sum"] = loss
            values[f"{name}_correct"] = int((logits.argmax(-1) == target).sum().item())
            ce_sums.append(loss)
        component_sum = sum(ce_sums) / 3.0
        values["component_ce_sum"] = component_sum
        values["objective_sum"] = values["objective_sum"] + component_sum
        predicted = torch.stack([outputs["component_logits"][n][mask].argmax(-1) for n in ("initials", "finals")], dim=-1)
        valid_pairs = torch.tensor(sorted(valid_initial_final_pairs()), dtype=torch.long, device=predicted.device)
        valid = (predicted[:, None, :] == valid_pairs[None, :, :]).all(dim=-1).any(dim=-1)
        values["invalid_if_count"] = int((~valid).sum().item())
        values["joint_ift_correct"] = int((torch.stack([outputs["component_logits"][n][mask].argmax(-1) for n in ("initials", "finals", "tones")], dim=-1) == labels).all(dim=-1).sum().item())
    if family in {"acoustic", "components-acoustic"}:
        target_ids = batch["acoustic_ids"][mask]
        prediction = outputs["acoustic_prediction"][mask]
        targets = acoustic_targets[target_ids]
        cosine_distance = 1.0 - F.cosine_similarity(prediction, targets, dim=-1, eps=COSINE_EPSILON)
        cosine_sum = cosine_distance.sum()
        values["cosine_sum"] = cosine_sum
        values["predicted_norm_below_epsilon"] = int((torch.linalg.vector_norm(prediction.float(), dim=-1) < COSINE_EPSILON).sum().item())
        weighted = (weight if family == "components-acoustic" else 1.0) * cosine_sum
        values["weighted_cosine_sum"] = weighted
        values["objective_sum"] = values["objective_sum"] + weighted
    values["objective"] = values["objective_sum"] / count
    return values


def finalize_epoch(totals: dict) -> dict:
    count = int(totals.get("characters", 0))
    if count <= 0:
        raise ValueError("loader contains no valid characters")
    result = {"characters": count, "objective": totals["objective_sum"] / count}
    for key, value in totals.items():
        if key.endswith("_sum") and key != "objective_sum":
            result[key.removesuffix("_sum")] = value / count
    for key in ("syllable_correct", "initials_correct", "finals_correct", "tones_correct", "joint_ift_correct"):
        if key in totals: result[key.removesuffix("_correct") + "_accuracy"] = totals[key] / count
    if "invalid_if_count" in totals:
        result["invalid_if_rate"] = totals["invalid_if_count"] / count
        result["invalid_if_count"] = totals["invalid_if_count"]
    if "predicted_norm_below_epsilon" in totals:
        result["predicted_norm_below_epsilon"] = totals["predicted_norm_below_epsilon"]
    return result


## Reproducibility

Use fixed seeds and separate random streams for shared encoder initialization, readout heads and batch ordering. RNG snapshots are saved without a resume interface.


In [ ]:
def seed_everything(seed: int, deterministic: bool = True) -> None:
    if type(seed) is not int or seed < 0:
        raise ValueError("seed must be a nonnegative integer")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.benchmark = False


def make_generator(seed: int, stream: int = 0) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(int(seed) + 1_000_003 * int(stream))
    return generator


def rng_snapshot(loader_generator: torch.Generator | None = None) -> dict:
    numpy_state = np.random.get_state()
    state = {
        "python": list(random.getstate()),
        "numpy": {"algorithm": numpy_state[0], "keys": numpy_state[1].tolist(),
                  "position": int(numpy_state[2]), "has_gauss": int(numpy_state[3]),
                  "cached_gaussian": float(numpy_state[4])},
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }
    if loader_generator is not None:
        state["dataloader"] = loader_generator.get_state()
    return state

def _construct_with_streams(config: dict, vocab: dict, model_id: str, seed: int) -> TrainingModel:
    """Initialize shared weights and heads from independent seed streams."""
    model_id = canonical_model_id(model_id)
    devices = []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed))
        encoder = SharedEncoder(len(vocab["ids"]["tokens"]))
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(int(seed) + 1_000_003)
        model = TrainingModel(encoder, model_id, vocab, int(config["architecture"]["acoustic_dim"]))
    return model


def build_encoder(config: dict, vocab: dict, model_id: str | None = None, seed: int = 0) -> TrainingModel:
    """Build a full model; the returned `.encode(batch)` is the shared export API."""
    architecture = config.get("architecture", {})
    if architecture.get("embedding_dim") != 64 or architecture.get("context_dim") != 128 or architecture.get("acoustic_dim", 0) <= 0:
        raise ValueError("experiment architecture violates encoder dimension contract")
    selected = model_id or config.get("model_id")
    if selected is None:
        raise ValueError("model_id is required to construct objective heads")
    return _construct_with_streams(config, vocab, selected, seed)


def shared_encoder_sha256(model: TrainingModel) -> str:
    import hashlib
    digest = hashlib.sha256()
    for name, tensor in sorted(model.encoder.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode("utf-8")); digest.update(str(value.dtype).encode("ascii"))
        digest.update(repr(tuple(value.shape)).encode("ascii")); digest.update(value.numpy().tobytes())
    return digest.hexdigest()


## Training and diagnostics

Compute character-weighted epoch statistics and measure global L2 gradient norms after backward and before the optimizer step. Stop on non-finite losses or gradients without clipping.


In [ ]:
class TrainingStepFailure(FloatingPointError):
    """A non-finite training step with JSON-safe diagnostic context."""

    def __init__(self, message, diagnostic):
        super().__init__(message)
        self.diagnostic = diagnostic


def _nonfinite_status(value):
    if math.isnan(value): return "nan"
    return "positive_infinity" if value > 0 else "negative_infinity"


def _gradient_summary(norm_values, nonfinite_count):
    return {"mean": statistics.fmean(norm_values) if norm_values else 0.0,
            "std": statistics.pstdev(norm_values) if len(norm_values) > 1 else 0.0,
            "max": max(norm_values) if norm_values else 0.0,
            "nonfinite_count": nonfinite_count}


def _failure_diagnostic(event, gradient_norms, nonfinite_count):
    """Retain successful work in the current epoch when a later step fails."""
    return {"nonfinite_count": int(nonfinite_count),
            "gradient_norms": list(gradient_norms), "event": event}


def _jsonable_metrics(values):
    return {key: (float(value.detach().cpu()) if torch.is_tensor(value) else value) for key, value in values.items()}


def _move(batch, device):
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}


def _gradient_norm(model):
    squared_norm = torch.zeros((), dtype=torch.float64, device="cpu")
    finite = True
    for parameter in model.parameters():
        if parameter.grad is None:
            continue
        grad = parameter.grad.detach()
        if not bool(torch.isfinite(grad).all()):
            finite = False
        squared_norm += grad.to(device="cpu", dtype=torch.float64).square().sum()
    value = float(torch.sqrt(squared_norm).item())
    return value, finite and math.isfinite(value)


def _epoch(model, loader, acoustic_targets, model_id, device, *, optimizer=None, event_writer=None,
           epoch_number=None, step_start=0):
    training = optimizer is not None
    model.train(training)
    totals = {}
    gradient_norms = []
    nonfinite_gradients = 0
    step = step_start
    for batch in loader:
        batch = _move(batch, device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            outputs = model(batch)
            stats = batch_objective(outputs, batch, acoustic_targets, model_id)
            objective = stats["objective"]
            if not bool(torch.isfinite(objective)):
                value = float(objective.detach().cpu())
                record = {"event": "nonfinite_loss", "epoch": epoch_number, "step": step,
                          "loss": None, "loss_status": _nonfinite_status(value)}
                if event_writer: event_writer(record)
                raise TrainingStepFailure(f"non-finite loss at epoch {epoch_number}, step {step}",
                                          _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
            if training:
                objective.backward()
                grad_norm, finite_grad = _gradient_norm(model)
                if not finite_grad:
                    nonfinite_gradients += 1
                    record = {"event": "nonfinite_gradient", "epoch": epoch_number, "step": step,
                              "gradient_norm": None, "gradient_norm_status": _nonfinite_status(grad_norm)}
                    if event_writer: event_writer(record)
                    raise TrainingStepFailure(f"non-finite gradient at epoch {epoch_number}, step {step}",
                                              _failure_diagnostic(record, gradient_norms, nonfinite_gradients))
                gradient_norms.append(grad_norm)
                optimizer.step()
            else:
                grad_norm = None
        detached = _jsonable_metrics(stats)
        for key, value in detached.items():
            if key == "objective":
                continue
            totals[key] = totals.get(key, 0) + value
        if event_writer:
            event_writer({"event": "step", "epoch": epoch_number, "step": step,
                          "losses_per_character": {key.removesuffix("_sum"): value / stats["characters"]
                                                     for key, value in detached.items() if key.endswith("_sum")},
                          "objective": detached["objective"], "gradient_norm": grad_norm})
        step += 1
    result = finalize_epoch(totals)
    result["steps"] = step - step_start
    result["gradient_norms"] = gradient_norms
    result["gradient_nonfinite_count"] = nonfinite_gradients
    return result, step


## Checkpoints and summaries

Train five seeds sequentially with Adam. Save the earliest best validation checkpoint, last-state snapshots, per-seed results and run summaries. File provenance uses the canonical hashes shown above.


In [ ]:
CODE_FILES = ("training.py", "model.py", "data.py", "metrics.py", "reproducibility.py", "experiment.py",
              "training_data.py", "objectives.py", "pinyin.py", "ids.py", "artifacts.py", "data_contract.py")


def _state_hash(state):
    digest = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode()); digest.update(str(value.dtype).encode()); digest.update(repr(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def _test_summary(seed_results):
    metrics = {}
    completed = [row for row in seed_results if "test_metrics" in row]
    keys = sorted(set.intersection(*(set(row["test_metrics"]) for row in completed))) if completed else []
    for key in keys:
        values = [float(row["test_metrics"][key]) for row in completed]
        metrics[key] = {"mean": statistics.fmean(values), "std": statistics.pstdev(values)}
    return metrics


def _write_run_manifest(run_dir, manifest):
    files = {}
    for file in sorted(run_dir.rglob("*")):
        if file.is_file() and file.name != "manifest.json":
            files[file.relative_to(run_dir).as_posix()] = sha256_file(file)
    manifest["artifacts"] = files
    write_json(run_dir / "manifest.json", manifest)


def train_experiment(experiment_path, *, model_id: str, runs_root=None, device: str | None = None,
                     run_id: str | None = None, expected_manifest_sha256: str | None = None):
    """Train the selected model over the five fixed seeds, sequentially, without resume."""
    model_id = canonical_model_id(model_id)
    _, lambda_cosine = parse_model_id(model_id)
    experiment = load_experiment(experiment_path)
    experiment_root = Path(experiment["root"])
    if expected_manifest_sha256 is not None:
        actual_manifest_sha256 = sha256_file(experiment_root / "manifest.json")
        if actual_manifest_sha256 != expected_manifest_sha256:
            raise ValueError("experiment manifest does not match the explicitly selected input pin")
    config = experiment["config"]
    seeds = config["training_seeds"]
    requested_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    target_device = torch.device(requested_device)
    if target_device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(f"selected device {requested_device!r} requires CUDA")
    experiment_id = experiment["manifest"]["experiment_id"]
    run_id = run_id or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8]
    if not isinstance(run_id, str) or not run_id or Path(run_id).name != run_id or "/" in run_id or "\\" in run_id:
        raise ValueError("run_id must be a safe path component")
    default_root = Path(OUTPUT_ROOT)
    output_root = Path(runs_root).resolve() if runs_root else default_root
    run_dir = output_root / experiment_id / model_id / run_id
    if run_dir.exists():
        raise FileExistsError(f"run already exists: {run_dir}")
    run_dir.parent.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=False, exist_ok=False)
    manifest = {
        "schema_version": 1, "run_type": "training_run", "status": "running", "run_id": run_id,
        "experiment_id": experiment_id, "model_id": model_id, "lambda_cosine": lambda_cosine, "device": str(target_device), "seeds": seeds,
        "selection": "minimum per-seed validation objective; strict improvement keeps earliest tie",
        "experiment_hashes": {name: sha256_file(experiment_root / name) for name in
                               ("manifest.json", "config.json", "vocab.json", "samples.jsonl", "splits.json", "acoustic_targets.npy")},
        "config": config, "code_sha256": {name: BUILD_METADATA["source_sha256"][name] for name in CODE_FILES},
        "runtime": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
                    "cuda_available": torch.cuda.is_available(), "cuda_version": torch.version.cuda},
        "seed_results": [], "summary": None, "artifacts": {},
    }
    write_json(run_dir / "manifest.json", manifest)
    seed_results = []
    statuses = {str(seed): "pending" for seed in seeds}
    global_step = 0
    active_seed = None
    try:
        for seed in seeds:
            active_seed = seed
            norm_values = []
            nonfinite_count = 0
            seed_dir = run_dir / f"seed-{seed}"
            seed_dir.mkdir()
            statuses[str(seed)] = "running"
            event_path = seed_dir / "events.jsonl"
            def log_event(event):
                with event_path.open("a", encoding="utf-8", newline="\n") as handle:
                    handle.write(json.dumps(event, sort_keys=True, allow_nan=False) + "\n")
            seed_everything(seed)
            model = build_encoder(config, experiment["vocab"], model_id=model_id, seed=seed).to(target_device)
            initial_state = {key: value.detach().cpu().clone() for key, value in model.encoder.state_dict().items()}
            shared_hash = _state_hash(initial_state)
            torch.save({"state_dict": initial_state, "seed": seed, "sha256": shared_hash}, seed_dir / "shared_encoder.pt")
            loaders = create_loaders(experiment, seed, batch_size=32, num_workers=0)
            acoustic = torch.as_tensor(experiment["acoustic_targets"], dtype=torch.float32, device=target_device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)
            best = math.inf
            best_epoch = None
            best_validation = None
            epoch_history = []
            for epoch_num in range(1, 21):
                train_metrics, global_step = _epoch(model, loaders["train"], acoustic, model_id, target_device,
                                                    optimizer=optimizer, event_writer=log_event,
                                                    epoch_number=epoch_num, step_start=global_step)
                norm_values.extend(train_metrics.pop("gradient_norms"))
                nonfinite_count += train_metrics.pop("gradient_nonfinite_count")
                validation_metrics, _ = _epoch(model, loaders["validation"], acoustic, model_id, target_device,
                                               event_writer=log_event, epoch_number=epoch_num)
                val_objective = float(validation_metrics["objective"])
                epoch_history.append({"epoch": epoch_num, "train": train_metrics, "validation": validation_metrics})
                if val_objective < best:
                    best, best_epoch, best_validation = val_objective, epoch_num, validation_metrics
                    torch.save({"state_dict": model.state_dict(), "model_id": model_id, "lambda_cosine": lambda_cosine, "seed": seed,
                                "best_epoch": best_epoch, "validation_objective": best,
                                "shared_encoder_sha256": shared_hash,
                                "experiment_hashes": manifest["experiment_hashes"],
                                "config_sha256": sha256_file(experiment_root / "config.json"),
                                "vocab_sha256": sha256_file(experiment_root / "vocab.json")}, seed_dir / "best.pt")
                torch.save({"state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                            "seed": seed, "epoch": epoch_num, "rng_state": rng_snapshot(loaders["train"].generator)},
                           seed_dir / "last.pt")
                write_json(seed_dir / "history.json", {"epochs": epoch_history})
                log_event({"event": "epoch", "epoch": epoch_num, "train": train_metrics, "validation": validation_metrics,
                           "best_epoch": best_epoch, "best_validation_objective": best})
            best_checkpoint = torch.load(seed_dir / "best.pt", map_location=target_device, weights_only=True)
            model.load_state_dict(best_checkpoint["state_dict"], strict=True)
            test_metrics, _ = _epoch(model, loaders["test"], acoustic, model_id, target_device,
                                     event_writer=log_event, epoch_number="test")
            test_metrics.pop("gradient_norms"); test_metrics.pop("gradient_nonfinite_count")
            grad_summary = _gradient_summary(norm_values, nonfinite_count)
            result = {"seed": seed, "status": "complete", "best_epoch": best_epoch,
                      "best_validation_objective": best, "best_validation_metrics": best_validation,
                      "test_metrics": test_metrics, "gradient_norm": grad_summary,
                      "shared_encoder_sha256": shared_hash}
            seed_results.append(result)
            statuses[str(seed)] = "complete"
            write_json(seed_dir / "summary.json", result)
            manifest["seed_results"] = seed_results
            manifest["seed_status"] = statuses
            write_json(run_dir / "summary.json", {"status": "running", "completed_seeds": len(seed_results),
                        "seed_results": seed_results, "test_metrics": _test_summary(seed_results)})
            _write_run_manifest(run_dir, manifest)
            active_seed = None
        gradient = [row["gradient_norm"] for row in seed_results]
        run_summary = {"seeds_completed": len(seed_results), "test_metrics": _test_summary(seed_results),
                       "gradient_norm": {"mean_of_seed_means": statistics.fmean(x["mean"] for x in gradient),
                                         "max_of_seed_maxima": max(x["max"] for x in gradient),
                                         "nonfinite_count": sum(x["nonfinite_count"] for x in gradient)}}
        manifest.update({"status": "complete", "seed_status": statuses, "seed_results": seed_results, "summary": run_summary})
        write_json(run_dir / "summary.json", {"status": "complete", **run_summary, "seed_results": seed_results})
        _write_run_manifest(run_dir, manifest)
        return {"run_dir": run_dir, "manifest": manifest, "summary": run_summary}
    except Exception as exc:
        failure_diagnostic = getattr(exc, "diagnostic", None)
        if active_seed is not None and statuses.get(str(active_seed)) == "running":
            prior_norms = list(locals().get("norm_values", []))
            prior_nonfinite = int(locals().get("nonfinite_count", 0))
            if failure_diagnostic:
                prior_norms.extend(failure_diagnostic.get("gradient_norms", []))
                prior_nonfinite += int(failure_diagnostic.get("nonfinite_count", 0))
            failed_result = {"seed": active_seed, "status": "failed",
                             "gradient_norm": _gradient_summary(prior_norms, prior_nonfinite)}
            if failure_diagnostic:
                failed_result["failure_event"] = failure_diagnostic["event"]
            seed_results.append(failed_result)
            statuses[str(active_seed)] = "failed"
            failed_dir = run_dir / f"seed-{active_seed}"
            if failed_dir.is_dir():
                write_json(failed_dir / "summary.json", failed_result)
        statuses.update({key: ("failed" if value == "running" else value) for key, value in statuses.items()})
        status = "partial" if seed_results else "failed"
        failure = {"exception": type(exc).__name__, "message": str(exc), "completed_seeds": sum(x["status"] == "complete" for x in seed_results),
                   "seed_status": statuses, "global_step": global_step,
                   "diagnostic": failure_diagnostic}
        write_json(run_dir / "failure.json", failure)
        manifest.update({"status": status, "seed_status": statuses, "seed_results": seed_results, "failure": failure})
        write_json(run_dir / "summary.json", {"status": status, "seed_results": seed_results,
                    "test_metrics": _test_summary(seed_results),
                    "gradient_norm": {"mean_of_seed_means": statistics.fmean(x["gradient_norm"]["mean"] for x in seed_results),
                                      "max_of_seed_maxima": max(x["gradient_norm"]["max"] for x in seed_results),
                                      "nonfinite_count": sum(x["gradient_norm"]["nonfinite_count"] for x in seed_results)}})
        _write_run_manifest(run_dir, manifest)
        raise


## Run training

Enable only after reviewing the pinned experiment, device and output path. This cell trains the selected objective for all five seeds.


In [ ]:
if RUN_REAL_TRAINING:
    result = train_experiment(
        EXPERIMENT_PATH, model_id=MODEL_ID, runs_root=OUTPUT_ROOT, device=DEVICE,
        expected_manifest_sha256=EXPECTED_EXPERIMENT_MANIFEST_SHA256,
    )
    result["summary"]
else:
    print("Training is disabled. Review the pinned experiment and settings before enabling it.")
